# 数据科学导论线性回归课后作业

* 本次作业的目标深入理解线性回归模型及其求解方法
* 包含**理论作业**和**编程作业**两部分，请均在Jupyter Notebook中答题
  * 可以在互联网中搜索Markdown语法，特别是Latex风格的公式写法
* 本次作业在OBE提交，截止时间为**11月9日 23:55**
  * 命名规则：Lab02-学号.ipynb，例子：Lab02-2020123456.ipynb

## 第一部分：理论作业

1. 给定一个多变量线性回归（MLR）模型$\hat{\mathbf{Y}}=\mathbf{X}\mathbf{\theta}$，请完成：
  * 写出损失函数均方误差（MSE）的矩阵形式$R(\mathbf{\theta})$;
    * 注：考虑分母有`2`的形式
  * 证明：令均方误差$R(\mathbf{\theta})$最小化的条件是$\mathbf{X}^𝑇 (\mathbf{Y}−\mathbf{X}\mathbf{\theta})=0$

请在此处答题：
* 损失函数均方误差（MSE）的矩阵形式$R(\mathbf{\theta})$ 为 $$ R(\boldsymbol{\theta}) = \frac{1}{2n} (\mathbf{Y} - \mathbf{X}\boldsymbol{\theta})^\top (\mathbf{Y} - \mathbf{X}\boldsymbol{\theta})$$ 

* 证明如下：
  * 计算梯度 $$ \nabla_{\boldsymbol{\theta}} R(\boldsymbol{\theta}) 
= \frac{1}{2n} \nabla_{\boldsymbol{\theta}} \left[ (\mathbf{Y} - \mathbf{X}\boldsymbol{\theta})^\top (\mathbf{Y} - \mathbf{X}\boldsymbol{\theta}) \right] $$  
  * 展开
$$
\nabla_{\boldsymbol{\theta}} R(\boldsymbol{\theta}) = \frac{1}{2n} \left( -2 \mathbf{X}^\top \mathbf{Y} + 2 \mathbf{X}^\top \mathbf{X} \boldsymbol{\theta} \right)
= \frac{1}{n} \left( -\mathbf{X}^\top \mathbf{Y} + \mathbf{X}^\top \mathbf{X} \boldsymbol{\theta} \right)
$$
  * 令梯度为 0
$$ \mathbf{X}^\top \mathbf{Y} + \mathbf{X}^\top \mathbf{X} \boldsymbol{\theta} = 0 $$
  * 即
$$ \mathbf{X}^\top (\mathbf{Y} - \mathbf{X} \boldsymbol{\theta}) = 0 $$

2. 给定一个多变量线性回归（MLR）模型$\hat{\mathbf{Y}}=\mathbf{X}\mathbf{\theta}$，请完成：
  * 请写出损失函数为上述均方误差（MSE）情况下的梯度函数$\nabla_{\mathbf{\theta}}{R(\mathbf{\theta})}$；
  * 如果进一步考虑了L2正则项，请写出梯度函数$\nabla_{\mathbf{\theta}}{R(\mathbf{\theta})}$
     * 注：采用$R(\mathbf{\theta}) + \frac{\lambda}{2} L_2(\theta)$的形式

请在此处答题：
* 由题1可知 
$$
\nabla_{\boldsymbol{\theta}} R(\boldsymbol{\theta}) = \frac{1}{n}(\mathbf{X}^\top (\mathbf{X}\boldsymbol{\theta} - \mathbf{Y}))
$$

* 考虑 L2 正则项后的损失函数为
$$
R_{\text{reg}}(\boldsymbol{\theta}) = \frac{1}{2n}(R(\boldsymbol{\theta}) + \frac{\lambda}{2} L_2(\boldsymbol{\theta})) = \frac{1}{2n}(R(\boldsymbol{\theta}) + \frac{\lambda}{2} \ \boldsymbol{\theta}\ ^2)
$$
梯度为：
$$
\nabla_{\boldsymbol{\theta}} R_{\text{reg}}(\boldsymbol{\theta}) = \frac{1}{n}(\mathbf{X}^\top (\mathbf{X}\boldsymbol{\theta} - \mathbf{Y}) + \lambda \boldsymbol{\theta})
$$

## 第二部分：编程作业 - 实现梯度下降与随机梯度下降

### 题目描述

In [2]:
# 确定随机数种子，确保记事本运行结果的确定性
import numpy as np
import pandas as pd
np.random.seed(43)

#### 考虑波士顿房价数据集
* 从`sklearn`库载入数据集
* 简单起见，本次作业仅考虑两个输入变量`RM`和`LSTAT`
* 目标变量为`PRICE`

In [ ]:
from sklearn.datasets import load_boston

boston = load_boston()
boston_df = pd.DataFrame(boston.data)
boston_df.columns = boston.feature_names
boston_df['PRICE'] = boston.target
X = boston_df[['RM','LSTAT']]
y = boston_df[['PRICE']]
X,y

#### 数据预处理：我们将每个输入变量都规范到`[0,1]`的区间内

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(X)
X=scaler.transform(X)
X

array([[0.57750527, 0.08967991],
       [0.5479977 , 0.2044702 ],
       [0.6943859 , 0.06346578],
       ...,
       [0.65433991, 0.10789183],
       [0.61946733, 0.13107064],
       [0.47307913, 0.16970199]])

#### 划分训练集与验证集，按照`4:1`的比例
* 简单起见，我们确定了随机数种子

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_train, y_test

(     PRICE
 477   12.0
 15    19.9
 332   19.4
 423   13.4
 19    18.2
 ..     ...
 106   19.5
 270   21.1
 348   24.5
 435   13.4
 102   18.6
 
 [404 rows x 1 columns],
      PRICE
 173   23.6
 274   32.4
 491   13.6
 72    22.8
 452   16.1
 ..     ...
 412   17.9
 436    9.6
 411   17.2
 86    22.5
 75    21.4
 
 [102 rows x 1 columns])

#### 我们首先看下`sklearn`库中的`LinearRegression`模型在**测试集**上的准确率

In [7]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression().fit(X_train, y_train)
reg.intercept_, reg.coef_

(array([-3.84117708]), array([[ 5.46509877, -0.63249856]]))

In [8]:
from sklearn.metrics import mean_squared_error

y_reg = reg.predict(X_test)
mean_squared_error(y_test, y_reg)

31.243290601783627

#### 接下来，我们在损失函数中L2正则化因子，即`Ridge`回归模型

In [9]:
from sklearn.linear_model import Ridge

rdg = Ridge(alpha=1.0).fit(X_train, y_train)
rdg.intercept_, reg.coef_

(array([-3.54088896]), array([[ 5.46509877, -0.63249856]]))

In [10]:
y_rdg = rdg.predict(X_test)
mean_squared_error(y_test, y_rdg)

31.195615711646912

### 请编程实现随机梯度下降函数`sgd`，以及训练函数`fit`、预测函数`predict`

#### 损失函数和梯度的实现应与理论作业中的公式相同

* 损失函数`mse_loss_lr`，正则化参数默认为`0`，即不考虑正则化

In [18]:
def mse_loss_lr(theta, X, y, alpha=0.0):
    Xb = np.c_[np.ones((X.shape[0], 1)), X]
    y = y.values.reshape(-1, 1)
    n = len(y)
    y_pred = Xb @ theta.reshape(-1, 1)
    loss = (1/(2*n)) * np.sum((y - y_pred)**2) + (alpha/2) * np.sum(theta[1:]**2)
    return loss

* **检查点1**：考察函数编写的正确性

In [19]:
theta1 = np.array([0, -50, -10])
mse_loss_lr(theta1, X_train, y_train, alpha=1.0)

109959.37018564357

* 梯度函数`gradient_mse_lr`：注意有可能考虑正则化（参见理论作业的第2题）

In [29]:
def gradient_mse_lr(X, y, theta, alpha=0.0):
    X = np.array(X)
    y = np.array(y).reshape(-1, 1)
    n = X.shape[0]
    Xb = np.c_[np.ones((n, 1)), X]
    theta = theta.reshape(-1, 1)
    grad = (1/n) * (Xb.T @ (Xb @ theta - y)) + alpha * np.r_[[0], theta[1:].flatten()].reshape(-1,1)
    return grad.flatten()

* **检查点2**：考察函数编写的正确性

In [21]:
gradient_mse_lr(X_train, y_train, theta1, 1.0)

array([ -463.16460396, -2974.26977376, -6081.25969678])

* 随机梯度下降函数`sgd`

In [30]:
def sgd(
    gradient, X, y, start, learn_rate=0.1, batch_size=1, n_iter=50,
    tolerance=1e-6, dtype="float64", random_state=None
):
    if random_state:
        np.random.seed(random_state)
    theta = np.array(start, dtype=dtype)
    n = X.shape[0]
    prev_theta = theta.copy()
    
    for _ in range(n_iter):
        indices = np.random.permutation(n)
        X_shuffled = X.iloc[indices] if isinstance(X, pd.DataFrame) else X[indices]
        y_shuffled = y.iloc[indices] if isinstance(y, pd.DataFrame) else y[indices]
        
        for i in range(0, n, batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]
            grad = gradient(X_batch, y_batch, theta)
            theta -= learn_rate * grad
        
        if np.linalg.norm(theta - prev_theta) < tolerance:
            break
        prev_theta = theta.copy()
    
    return theta

* 基于上面实现的四个函数，实现训练函数`fit`
  * 输入为训练数据`X_train`和`y_train`
  * 输出为估计的参数`theta_hat`
  * 可以根据数据自由地**炼丹**，即决定初始点、超参数、迭代次数等

In [31]:
def fit(X_train, y_train):
    start = np.zeros(X_train.shape[1] + 1)
    theta_hat = sgd(
        gradient_mse_lr,
        X_train, y_train,
        start=start,
        learn_rate=0.001, 
        batch_size=8,
        n_iter=1000,
        random_state=42
    )
    return theta_hat

* 基于训练出的模型，实现预测函数`predict`
  * 输入为测试数据`X_test`
  * 输出为预测值`y_test`

In [25]:
def predict(X_test, theta_hat):
    Xb = np.c_[np.ones((X_test.shape[0], 1)), X_test]
    y_pred = Xb @ theta_hat.reshape(-1, 1)
    return y_pred

### 评测与调优
* 基于以下评测代码，可以反复调优；
* **检查点3**：分析所得出的RMSE与`sklearn`给出的RMSE的差异性，检查分析的合理性与深度

In [32]:
theta_hat = fit(X_train, y_train)
y_prd = predict(X_test, theta_hat)
mean_squared_error(y_test, y_prd)

31.56596028746431